<a href="https://colab.research.google.com/github/halkeralmasri/first-peojict-in-ML-model/blob/main/ML%20model%20for%20a%20pridect%20the%20price%20of%20car's.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# import the liplary
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error

In [ ]:
#load data
url = "https://raw.githubusercontent.com/YBIFoundation/Dataset/main/Car%20Price.csv"
df = pd.read_csv(url)

In [ ]:
# just for a more details
print(" --- information for a data ---")
print(df.info())

print("/n --- Statistical details ---")
print(df.describe())

 --- information for a data ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4340 entries, 0 to 4339
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Brand          4340 non-null   object
 1   Model          4340 non-null   object
 2   Year           4340 non-null   int64 
 3   Selling_Price  4340 non-null   int64 
 4   KM_Driven      4340 non-null   int64 
 5   Fuel           4340 non-null   object
 6   Seller_Type    4340 non-null   object
 7   Transmission   4340 non-null   object
 8   Owner          4340 non-null   object
dtypes: int64(3), object(6)
memory usage: 305.3+ KB
None
/n --- Statistical details ---
              Year  Selling_Price      KM_Driven
count  4340.000000   4.340000e+03    4340.000000
mean   2013.090783   5.041273e+05   66215.777419
std       4.215344   5.785487e+05   46644.102194
min    1992.000000   2.000000e+04       1.000000
25%    2011.000000   2.087498e+05   35000.000000
50%    

In [ ]:
#---------------------------------------------------
#-- Exploratory Cleaning data  --
#---------------------------------------------------
# Remove completely duplicate rows
df.drop_duplicates(inplace=True)

#(Feature Engineering)
df['Car_Age'] = 2026 - df['Year']   # just for calculat the age of car

# Removing redundant or non-useful columns for direct prediction
df.drop(columns=['Year', 'Brand', 'Model'], inplace=True, errors='ignore')

# (Features & Target)
X = df.drop(columns=['Selling_Price'])
y = df['Selling_Price']

# Automatically determine column classification
num_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_features = X.select_dtypes(include=['object']).columns.tolist()
# teast for if the Exploratory Cleaning data is done
print ("Exploratory Cleaning data is done ")

Exploratory Cleaning data is done 


In [ ]:
# ---------------------------------------------------------
# ----Setting up the professional processing pipeline-----
# ---------------------------------------------------------

# Pipeline for processing numerical columns
numeric_transformer = Pipeline(steps=[
('imputer', SimpleImputer(strategy='median')),  # Impute missing values ​​with the median
('scaler', RobustScaler())                      # Scale data and handle outliers
])

# Pipeline for processing categorical columns
categorical_transformer = Pipeline(steps=[
('imputer', SimpleImputer(strategy='most_frequent')), # Impute missing values ​​with the most frequent value
('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)) # Apply One-Hot Encoding
])

# Combining numerical and categorical processing
preprocessor = ColumnTransformer(transformers=[
('num', numeric_transformer, num_features),
('cat', categorical_transformer, cat_features)
])

In [ ]:
# ---------------------------------------------------------
# --Integrating preprocessing with the model and training--
# ---------------------------------------------------------

# Building the final pipeline
full_pipeline = Pipeline(steps=[
('preprocessor', preprocessor),
('model', RandomForestRegressor(n_estimators=200, random_state=42))
])

# Splitting the data (80% training / 20% testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Training the full pipeline (preprocessing + model)
full_pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   RobustScaler())]),
                                                  ['KM_Driven', 'Car_Age']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['Fuel', 'Seller_Type',
                                                   'Transmission',
                                                   'Owner'])])),
                ('model',
                 RandomForestRegressor(n_estimators=200, random_state=42))])

In [ ]:
# Evaluation
y_pred = full_pipeline.predict(X_test)
print(f"R² Score: {r2_score(y_test, y_pred):.4f}")
print(f"MAE: ${mean_absolute_error(y_test, y_pred):,.2f}")

R² Score: 0.4446
MAE: $192,403.20


In [ ]:
# and the advanced cod for ge a best R^2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler, OneHotEncoder, TargetEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# 1. Load Data
url = "https://raw.githubusercontent.com/YBIFoundation/Dataset/main/Car%20Price.csv"
df = pd.read_csv(url)

# Remove duplicates and feature engineering
df.drop_duplicates(inplace=True)
df['Car_Age'] = 2026 - df['Year']
df.drop(columns=['Year'], inplace=True, errors='ignore')

# 2. Remove extreme price outliers
q_high = df['Selling_Price'].quantile(0.98)
df = df[df['Selling_Price'] <= q_high]

X = df.drop(columns=['Selling_Price'])
# Apply Log1p to price to improve stability
y_log = np.log1p(df['Selling_Price'])

# 3. Define column types
high_card_cols = ['Brand', 'Model'] if 'Brand' in X.columns else []
low_card_cols = [col for col in X.select_dtypes(include=['object']).columns if col not in high_card_cols]
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

# 4. Set up Preprocessing Pipeline
num_transformer = Pipeline([
('imputer', SimpleImputer(strategy='median')),
('scaler', RobustScaler())
])

low_cat_transformer = Pipeline([
('imputer', SimpleImputer(strategy='most_frequent')),
('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

high_cat_transformer = Pipeline([
('imputer', SimpleImputer(strategy='most_frequent')),
('target', TargetEncoder(smooth="auto"))
])

preprocessor = ColumnTransformer(transformers=[
('num', num_transformer, num_cols),
('low_cat', low_cat_transformer, low_card_cols),
('high_cat', high_cat_transformer, high_card_cols)
])

# 5. Final Pipeline with Random Forest
model_pipeline = Pipeline([
('preprocessor', preprocessor),
('regressor', RandomForestRegressor(n_estimators=300, max_depth=12, min_samples_split=5, random_state=42))
])

# 6. Splitting and Training
X_train, X_test, y_train_log, y_test_log = train_test_split(X, y_log, test_size=0.2, random_state=42)
model_pipeline.fit(X_train, y_train_log)

# 7. Prediction and converting results back to original dollar values ​​(Exponential)
y_pred_log = model_pipeline.predict(X_test)
y_pred = np.expm1(y_pred_log)
y_test = np.expm1(y_test_log)

# 8. Calculating Final Evaluation
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print(f"New R² Score: {r2:.4f} ({r2*100:.2f}%)")
print(f"MAE (Mean Absolute Error): ${mae:,.2f}")

New R² Score: 0.6980 (69.80%)
MAE (Mean Absolute Error): $104,257.31


In [ ]:
#--------------------------------------------------
# Final Note: Please note that this code is meant for a small portfolio project and baseline demonstration. Thank you for your time! :)